In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
NVIDIA GeForce RTX 5050 Laptop GPU
13.0


In [10]:
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os
import datetime


load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")


# 1. Charger le DataFrame filtré
df = pd.read_csv(
    "b_hashed_list.csv",
    usecols=[
        "title",
        # "ref",
        "status",
        "CELEX number",
        "Author",
        "Date of document",
        "link",
        # "Latest consolidated version",
        "hash_id"
    ]
)


#clean & prepare metadata
df = df.set_index("hash_id")
df["language"] = "english"
df["data source"] = "eur-lex"
df["date of migration"] = str(datetime.datetime.now().date())
df["Date of document"] = (
    df["Date of document"]
    .astype(str)
    .str.replace(r"[:;].*$", "", regex=True)
    .str.strip()
)
df.columns = df.columns.str.replace(":", "")
df.to_csv("b_hashed_list.csv")

In [11]:
df.columns

Index(['title', 'status', 'CELEX number', 'Author', 'Date of document', 'link',
       'language', 'data source', 'date of migration'],
      dtype='object')

In [12]:
print(qdrant_url)

https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/


In [ ]:
# 2. Ajout métadonnées → Document
def add_file_metadata(path):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}

models = {
    "sentence-transformers/all-MiniLM-L6-v2": 256,
    "sentence-transformers/all-MiniLM-L12-v2": 256,
    "Alibaba-NLP/Qwen3-Embedding-0.6B": 32768,
    "BAAI/bge-m3": 8194
}
# 3. Charger documents
documents = SimpleDirectoryReader(
    "xml_dir/",
    file_metadata=add_file_metadata
).load_data()

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(documents)


# 4. Embedding model
embed_model = HuggingFaceEmbedding(
    model_name= "BAAI/bge-m3",
    device="cuda",
    trust_remote_code=True
)


# 5. Qdrant setup
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
collection_name = "legal_"+"BAAI/bge-m3".replace('/', '_')
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(nodes, storage_context=storage_context, embed_model=embed_model)

2025-11-26 21:57:00,057 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\alaa-\miniconda3\envs\tekno\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alaa-\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

2025-11-26 22:08:07,251 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/ "HTTP/1.1 200 OK"
2025-11-26 22:08:07,561 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/exists "HTTP/1.1 200 OK"


AttributeError: type object 'VectorStoreIndex' has no attribute 'from_nodes'

In [14]:
print(storage_context)

StorageContext(docstore=<llama_index.core.storage.docstore.simple_docstore.SimpleDocumentStore object at 0x000001F35F5B9270>, index_store=<llama_index.core.storage.index_store.simple_index_store.SimpleIndexStore object at 0x000001F5A933AAD0>, vector_stores={'default': QdrantVectorStore(stores_text=True, is_embedding_query=True, flat_metadata=False, collection_name='legal_BAAI_bge-m3', url=None, api_key=None, batch_size=64, parallel=1, max_retries=3, client_kwargs={}, enable_hybrid=False, index_doc_id=True, fastembed_sparse_model=None, text_key='text', dense_vector_name='text-dense', sparse_vector_name='text-sparse-new'), 'image': SimpleVectorStore(stores_text=False, is_embedding_query=True, data=SimpleVectorStoreData(embedding_dict={}, text_id_to_ref_doc_id={}, metadata_dict={}))}, graph_store=<llama_index.core.graph_stores.simple.SimpleGraphStore object at 0x000001F5A933A860>, property_graph_store=None)


In [15]:
index = VectorStoreIndex.from_documents(nodes, storage_context=storage_context, embed_model=embed_model)

2025-11-26 22:15:32,127 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"
2025-11-26 22:15:32,545 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/index?wait=true "HTTP/1.1 200 OK"
2025-11-26 22:15:32,801 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"
2025-11-26 22:15:35,033 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points?wait=true "HTTP/1.1 200 OK"
2025-11-26 22:15:35,671 - INFO - HTTP Request: PUT https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points?wait=true "HTTP/1.1 200 OK"
2025-11-26 22:15:36,221 - INFO

ResponseHandlingException: The read operation timed out

In [22]:
from qdrant_client import QdrantClient
from tqdm import tqdm
import time

# 1. Connexion avec timeout augmenté
client = QdrantClient(
    url=qdrant_url, 
    api_key=qdrant_api_key,
    timeout=600  # 10 minutes
)

collection_name = "legal_BAAI_bge-m3"

# 2. Récupérer TOUS les IDs déjà dans Qdrant
print("📥 Récupération des IDs existants...")
qdrant_ids = set()
offset = None

while True:
    result, offset = client.scroll(
        collection_name=collection_name,
        limit=10000,  # Maximum par requête
        offset=offset,
        with_payload=False,
        with_vectors=False
    )
    qdrant_ids.update([point.id for point in result])
    
    if offset is None:
        break

print(f"✓ {len(qdrant_ids)} IDs trouvés dans Qdrant")

# 3. Filtrer les nodes déjà insérés
missing_nodes = [node for node in nodes if node.node_id not in qdrant_ids]
print(f"❌ {len(missing_nodes)} nodes manquants à insérer")
print(f"✅ {len(nodes) - len(missing_nodes)} nodes déjà présents")

# 4. Configuration optimisée pour reprendre
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="cuda",
    trust_remote_code=True,
    embed_batch_size=32  # ← CRITIQUE pour la vitesse
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    batch_size=5,      # Plus petit pour éviter timeout
    parallel=1,
    max_retries=5
)

# 5. Insérer les nodes manquants par lots avec retry
batch_size = 30
total_batches = (len(missing_nodes) + batch_size - 1) // batch_size

print(f"\n🚀 Insertion de {len(missing_nodes)} nodes en {total_batches} batches...")

for i in tqdm(range(0, len(missing_nodes), batch_size)):
    batch = missing_nodes[i:i+batch_size]
    batch_num = i // batch_size + 1
    
    # Retry avec backoff exponentiel
    for attempt in range(5):
        try:
            vector_store.add(batch)
            break  # Succès
        except Exception as e:
            if attempt < 4:
                wait_time = 5 * (2 ** attempt)  # 5s, 10s, 20s, 40s
                print(f"\n⚠️  Batch {batch_num} échoué (tentative {attempt+1}/5)")
                print(f"   Attente de {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"\n❌ Batch {batch_num} échoué après 5 tentatives")
                print(f"   Dernier index : {i}")
                print(f"   Vous pouvez reprendre avec : missing_nodes[{i}:]")
                raise

# 6. Vérification finale
final_count = client.get_collection(collection_name).points_count
print(f"\n{'='*60}")
print(f"📊 RÉSULTAT FINAL")
print(f"{'='*60}")
print(f"Points dans Qdrant : {final_count}/{len(nodes)}")

if final_count == len(nodes):
    print("✅ INSERTION COMPLÈTE ! 🎉")
else:
    print(f"⚠️  Encore {len(nodes) - final_count} nodes manquants")

2025-11-27 00:46:28,531 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/ "HTTP/1.1 200 OK"


📥 Récupération des IDs existants...


2025-11-27 00:46:28,763 - INFO - HTTP Request: POST https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points/scroll "HTTP/1.1 200 OK"
2025-11-27 00:46:29,147 - INFO - HTTP Request: POST https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points/scroll "HTTP/1.1 200 OK"
2025-11-27 00:46:29,436 - INFO - HTTP Request: POST https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points/scroll "HTTP/1.1 200 OK"
2025-11-27 00:46:29,674 - INFO - HTTP Request: POST https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points/scroll "HTTP/1.1 200 OK"
2025-11-27 00:46:29,907 - INFO - HTTP Request: POST https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/points/scroll "HTTP/1.1 200 OK"
2025-

✓ 48192 IDs trouvés dans Qdrant
❌ 76617 nodes manquants à insérer
✅ 0 nodes déjà présents


2025-11-27 00:46:40,145 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3/exists "HTTP/1.1 200 OK"
2025-11-27 00:46:40,221 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"


...

  0%|          | 0/2554 [00:00<?, ?it/s]


⚠️  Batch 1 échoué (tentative 1/5)
   Attente de 5s...

⚠️  Batch 1 échoué (tentative 2/5)
   Attente de 10s...

⚠️  Batch 1 échoué (tentative 3/5)
   Attente de 20s...

⚠️  Batch 1 échoué (tentative 4/5)
   Attente de 40s...


  0%|          | 0/2554 [01:15<?, ?it/s]


❌ Batch 1 échoué après 5 tentatives
   Dernier index : 0
   Vous pouvez reprendre avec : missing_nodes[0:]


ValueError: embedding not set.

In [25]:
from qdrant_client import QdrantClient

# --- Fonction utilitaire pour la conversion de taille ---
def convert_bytes(size_bytes):
    """Convertit les octets en format lisible (KB, MB, GB)."""
    if size_bytes == 0:
        return "0 B"
    size_name = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
    i = 0
    # Utilisation de 1024 pour les conversions binaires
    while size_bytes >= 1024 and i < len(size_name) - 1: 
        size_bytes /= 1024
        i += 1
    return f"{size_bytes:.2f} {size_name[i]}"
# --------------------------------------------------------

# Supposons que 'client' est déjà initialisé
# client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key, timeout=600)

total_size_bytes = 0
total_size_readable = ""

print("📦 Calcul du poids de la base de données Qdrant...")

try:
    # 1. Lister toutes les collections
    collections_response = client.get_collections()
    collection_names = [c.name for c in collections_response.collections]
    
    if not collection_names:
        print("❌ Aucune collection trouvée.")
        
    for name in collection_names:
        # 2. Récupérer les statistiques
        collection_info = client.get_collection(collection_name=name)
        
        # 3. Accéder à la taille sur disque (CORRECTION APPLIQUÉE ICI)
        # On accède directement à .config, car 'result' n'existe plus sur l'objet CollectionInfo
        size_bytes = collection_info.config.optimizer_config.disk_size_bytes 
        
        # 4. Accumuler et afficher le résultat par collection
        if size_bytes is not None:
            total_size_bytes += size_bytes
            print(f"  - {name}: {convert_bytes(size_bytes)}")
        else:
            print(f"  - {name}: Taille sur disque non disponible (N/A).")
            
    # 5. Afficher le total
    total_size_readable = convert_bytes(total_size_bytes)

    print("\n" + "="*50)
    print(f"✅ POIDS TOTAL DE LA BASE DE DONNÉES : {total_size_readable}")
    print("="*50)

except Exception as e:
    print(f"\n❌ Une erreur s'est produite lors de l'accès à Qdrant : {e}")

📦 Calcul du poids de la base de données Qdrant...


2025-11-27 00:56:18,554 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"
2025-11-27 00:56:18,648 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"



❌ Une erreur s'est produite lors de l'accès à Qdrant : 'OptimizersConfig' object has no attribute 'disk_size_bytes'


In [21]:
final_count = client.get_collection(collection_name).points_count
print(f"Points dans Qdrant : {final_count}/{len(nodes)}")

2025-11-27 00:40:47,393 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"


Points dans Qdrant : 48192/76617


In [16]:
# Après l'erreur ReadTimeout
collection_info = client.get_collection(collection_name=collection_name)
current_count = collection_info.points_count # Récupère le nombre actuel de points
print(f"Nombre actuel de points dans la collection : {current_count}")

2025-11-27 00:13:42,719 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"


Nombre actuel de points dans la collection : 48192


In [17]:
# Nombre de nodes générés par le splitter
total_nodes = len(nodes)
print(f"Nombre de nodes créés : {total_nodes}")

Nombre de nodes créés : 76617


In [18]:
# Nombre total de chunks à insérer
total_nodes = len(nodes)
print(f"Nodes à insérer : {total_nodes}")

# Nombre dans Qdrant
collection_info = client.get_collection(collection_name=collection_name)
print(f"Points dans Qdrant : {collection_info.points_count}")

# Vérifier si complet
if collection_info.points_count == total_nodes:
    print("✓ Insertion complète")
else:
    print(f"✗ Manquant : {total_nodes - collection_info.points_count} points")

Nodes à insérer : 76617


2025-11-27 00:28:02,545 - INFO - HTTP Request: GET https://225727c0-a3fe-4fe6-b234-d12cdee91e7b.europe-west3-0.gcp.cloud.qdrant.io:6333/collections/legal_BAAI_bge-m3 "HTTP/1.1 200 OK"


Points dans Qdrant : 48192
✗ Manquant : 28425 points


In [19]:
# 1. Vérifier un node aléatoire
sample_node = nodes[0]
print(f"Node ID : {sample_node.node_id}")
print(f"A un embedding ? {sample_node.embedding is not None}")

if sample_node.embedding is not None:
    print(f"Dimension : {len(sample_node.embedding)}")
    print(f"Premiers valeurs : {sample_node.embedding[:5]}")
else:
    print("❌ Pas d'embedding")

# 2. Vérifier tous les nodes
nodes_with_embeddings = sum(1 for node in nodes if node.embedding is not None)
nodes_without_embeddings = len(nodes) - nodes_with_embeddings

print(f"\n📊 Statistiques:")
print(f"✅ Nodes avec embedding : {nodes_with_embeddings}/{len(nodes)}")
print(f"❌ Nodes sans embedding : {nodes_without_embeddings}/{len(nodes)}")

if nodes_with_embeddings == len(nodes):
    print("🎉 Tous les nodes ont leurs embeddings!")
elif nodes_with_embeddings == 0:
    print("⚠️  Aucun node n'a d'embedding")
else:
    print("⚠️  Embeddings partiels")

Node ID : c541fc02-dfff-4e0a-9aa8-aaf5cd14cb0b
A un embedding ? False
❌ Pas d'embedding

📊 Statistiques:
✅ Nodes avec embedding : 0/76617
❌ Nodes sans embedding : 76617/76617
⚠️  Aucun node n'a d'embedding


In [ ]:
from qdrant_client import QdrantClient
from tqdm import tqdm
import time

# 1. Connexion avec timeout augmenté
client = QdrantClient(
    url=qdrant_url, 
    api_key=qdrant_api_key,
    timeout=600  # 10 minutes
)

collection_name = "legal_BAAI_bge-m3"

# 2. Récupérer TOUS les IDs déjà dans Qdrant
print("📥 Récupération des IDs existants...")
qdrant_ids = set()
offset = None

while True:
    result, offset = client.scroll(
        collection_name=collection_name,
        limit=10000,  # Maximum par requête
        offset=offset,
        with_payload=False,
        with_vectors=False
    )
    qdrant_ids.update([point.id for point in result])
    
    if offset is None:
        break

print(f"✓ {len(qdrant_ids)} IDs trouvés dans Qdrant")

# 3. Filtrer les nodes déjà insérés
missing_nodes = [node for node in nodes if node.node_id not in qdrant_ids]
print(f"❌ {len(missing_nodes)} nodes manquants à insérer")
print(f"✅ {len(nodes) - len(missing_nodes)} nodes déjà présents")

# 4. Configuration optimisée pour reprendre
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="cuda",
    trust_remote_code=True,
    embed_batch_size=32  # ← CRITIQUE pour la vitesse
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    batch_size=30,      # Plus petit pour éviter timeout
    parallel=1,
    max_retries=5
)

# 5. Insérer les nodes manquants par lots avec retry
batch_size = 30
total_batches = (len(missing_nodes) + batch_size - 1) // batch_size

print(f"\n🚀 Insertion de {len(missing_nodes)} nodes en {total_batches} batches...")

for i in tqdm(range(0, len(missing_nodes), batch_size)):
    batch = missing_nodes[i:i+batch_size]
    batch_num = i // batch_size + 1
    
    # Retry avec backoff exponentiel
    for attempt in range(5):
        try:
            vector_store.add(batch)
            break  # Succès
        except Exception as e:
            if attempt < 4:
                wait_time = 5 * (2 ** attempt)  # 5s, 10s, 20s, 40s
                print(f"\n⚠️  Batch {batch_num} échoué (tentative {attempt+1}/5)")
                print(f"   Attente de {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"\n❌ Batch {batch_num} échoué après 5 tentatives")
                print(f"   Dernier index : {i}")
                print(f"   Vous pouvez reprendre avec : missing_nodes[{i}:]")
                raise

# 6. Vérification finale
final_count = client.get_collection(collection_name).points_count
print(f"\n{'='*60}")
print(f"📊 RÉSULTAT FINAL")
print(f"{'='*60}")
print(f"Points dans Qdrant : {final_count}/{len(nodes)}")

if final_count == len(nodes):
    print("✅ INSERTION COMPLÈTE ! 🎉")
else:
    print(f"⚠️  Encore {len(nodes) - final_count} nodes manquants")

In [ ]:
import os
import pandas as pd
from pathlib import Path

from llama_index.core import SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.mistralai import MistralEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


# =========================
# CONFIGURATION
# =========================

# Variables d'environnement requises :
# export MISTRAL_API_KEY="xxx"
# export QDRANT_URL="http://localhost:6333"
# export QDRANT_API_KEY="xxx"

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY")
qdrant_url = os.environ.get("QDRANT_URL")
qdrant_api_key = os.environ.get("QDRANT_API_KEY")

# Charger ton dataframe contenant les métadonnées
df = pd.read_csv("metadata.csv", index_col=0)


# =========================
# 1. Ajout métadonnées → Document
# =========================

def add_file_metadata(path: str):
    p = Path(path)
    hash_id = p.stem
    row = df.loc[hash_id].to_dict()
    return {"hash_id": hash_id, **row}


# =========================
# 2. Charger les documents
# =========================

documents = SimpleDirectoryReader(
    "texts/",
    file_metadata=add_file_metadata
).load_data()


# =========================
# 3. Découpage en chunks
# =========================

splitter = SentenceSplitter(
    chunk_size=2048,
    chunk_overlap=50
)

nodes = splitter.get_nodes_from_documents(documents)


# =========================
# 4. Modèle d'embedding Mistral
# =========================

embed_model = MistralEmbedding(
    model="mistral-embed"
)

Settings.embed_model = embed_model


# =========================
# 5. Qdrant setup
# =========================

client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="eurlex"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


# =========================
# 6. Indexation
# =========================

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

print("✅ Indexation terminée avec Mistral embeddings")
